In [1]:
import random
import numpy as np
from data_process import get_CIFAR10_data
import math
from scipy.spatial import distance
from models import KNN, Perceptron, SVM, Softmax
from kaggle_submission import output_submission_csv
%matplotlib inline

Training data shape:  (50000, 32, 32, 3)
Training labels shape:  (50000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


# Loading CIFAR-10

In the following cells we determine the number of images for each split and load the images.

In [2]:
# You can change these numbers for experimentation
# For submission we will use the default values 
TRAIN_IMAGES = 49000
VAL_IMAGES = 1000
TEST_IMAGES = 5000

In [3]:
data = get_CIFAR10_data(TRAIN_IMAGES, VAL_IMAGES, TEST_IMAGES)
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

Training data shape:  (50000, 32, 32, 3)
Training labels shape:  (50000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


Convert the sets of images from dimensions of **(N, 3, 32, 32) -> (N, 3072)** where N is the number of images so that each **3x32x32** image is represented by a single vector.

In [4]:
X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_val = np.reshape(X_val, (X_val.shape[0], -1))
X_test = np.reshape(X_test, (X_test.shape[0], -1))

### Get Accuracy

This function computes how well your model performs using accuracy as a metric.

In [5]:
def get_acc(pred, y_test):
    return np.sum(y_test==pred)/len(y_test)*100

# K-Nearest Neighbors

The kNN classifier consists of two stages:

- During training, the classifier takes the training data and simply remembers it
- During testing, kNN classifies every test image by comparing to all training images and selecting the class that is most common among the k most similar training examples

In this exercise you will implement these steps using writing efficient, vectorized code. Your final implementation should not use for loops to loop over each of the test and train examples. Instead, you should calculate distances between vectorized forms of the datasets. You may refer to the `scipy.spatial.distance.cdist` function to do this efficiently.

The following code :
- Creates an instance of the KNN classifier class with k = 5
- The train function of the KNN class is trained on the training data
- We use the predict function for predicting testing data labels

### Training KNN

In [6]:
knn = KNN(1)
knn.train(X_train, y_train)

### Find best k on validation

The value of k is an important hyperparameter for the KNN classifier. We will choose the best k by examining the performance of classifiers trained with different k values on the validation set.

It's not necessary to try many different values of k for the purposes of this exercise. You may increase k by a magnitude of 2 each iteration up to around k=100 or something similar to get a sense of classifier performance for different k values.

**Modify the code below to loop though different values of k, train a KNN classifier for each k, and output the validation accuracy for each of the classifiers. Be sure to note your best k below as well.**

In [7]:
# TO DO : Experiment with different values of k
k = 5
knn = KNN(k)
knn.train(X_train, y_train)

pred_knn = knn.predict(X_val)
print('The validation accuracy is given by : %f' % (get_acc(pred_knn, y_val)))

The validation accuracy is given by : 32.600000


In [6]:
k_values = [1, 3, 5, 7, 9, 15, 25, 35, 50, 75, 100]

best_k = None
best_acc = 0

for k in k_values:
    print("Training with k=", k)
    
    knn = KNN(k)
    knn.train(X_train, y_train)
    
    pred_knn = knn.predict(X_val)
    
    acc = get_acc(pred_knn, y_val)
    
    print("Validation accuracy:", acc)
    
    if acc > best_acc:
        best_acc = acc
        best_k = k

print("Best k:", best_k)
print("Best validation accuracy:", best_acc)


Training with k= 1
Validation accuracy: 35.699999999999996
Training with k= 3
Validation accuracy: 32.7
Training with k= 5
Validation accuracy: 32.6
Training with k= 7
Validation accuracy: 32.300000000000004
Training with k= 9
Validation accuracy: 30.8
Training with k= 15
Validation accuracy: 30.8
Training with k= 25
Validation accuracy: 30.4
Training with k= 35
Validation accuracy: 29.799999999999997
Training with k= 50
Validation accuracy: 28.9
Training with k= 75
Validation accuracy: 28.999999999999996
Training with k= 100
Validation accuracy: 28.599999999999998
Best k: 1
Best validation accuracy: 35.699999999999996


### Testing KNN

Finally, once you have found the best k according to your experiments on the validation set, retrain a classifier with the best k and test your classifier on the test set.

In [7]:
best_k = 1 #As we got k=1 in above cell even though it is not a good choice for k(=1 will likely overfits), we will use it for testing as per the assignment requirement.
knn = KNN(best_k)
knn.train(X_train, y_train)

In [8]:
pred_knn = knn.predict(X_test)
print('The testing accuracy is given by : %f' % (get_acc(pred_knn, y_test)))

The testing accuracy is given by : 35.000000


### KNN Kaggle Submission

Once you are satisfied with your solution and test accuracy output a file to submit your test set predictions to the Kaggle for Assignment 1 KNN. Use the following code to do so:

In [9]:
output_submission_csv('knn_submission.csv', knn.predict(X_test))

# Perceptron

Perceptron has 2 hyperparameters that you can experiment with:
- **Learning rate** - controls how much we change the current weights of the classifier during each update. We set it at a default value of 0.5, but you should experiment with different values. We recommend changing the learning rate by factors of 10 and observing how the performance of the classifier changes. You should also try adding a **decay** which slowly reduces the learning rate over each epoch.
- **Number of Epochs** - An epoch is a complete iterative pass over all of the data in the dataset. During an epoch we predict a label using the classifier and then update the weights of the classifier according the perceptron update rule for each sample in the training set. You should try different values for the number of training epochs and report your results.

You will implement the Perceptron classifier in the **models/Perceptron.py**

The following code: 
- Creates an instance of the Perceptron classifier class 
- The train function of the Perceptron class is trained on the training data
- We use the predict function to find the training accuracy as well as the testing accuracy


### Train Perceptron

In [10]:
percept_ = Perceptron()
percept_.train(X_train, y_train)

Epoch 1/40 | lr=0.01000
Epoch 2/40 | lr=0.00990
Epoch 3/40 | lr=0.00980
Epoch 4/40 | lr=0.00971
Epoch 5/40 | lr=0.00962
Epoch 6/40 | lr=0.00952
Epoch 7/40 | lr=0.00943
Epoch 8/40 | lr=0.00935
Epoch 9/40 | lr=0.00926
Epoch 10/40 | lr=0.00917
Epoch 11/40 | lr=0.00909
Epoch 12/40 | lr=0.00901
Epoch 13/40 | lr=0.00893
Epoch 14/40 | lr=0.00885
Epoch 15/40 | lr=0.00877
Epoch 16/40 | lr=0.00870
Epoch 17/40 | lr=0.00862
Epoch 18/40 | lr=0.00855
Epoch 19/40 | lr=0.00847
Epoch 20/40 | lr=0.00840
Epoch 21/40 | lr=0.00833
Epoch 22/40 | lr=0.00826
Epoch 23/40 | lr=0.00820
Epoch 24/40 | lr=0.00813
Epoch 25/40 | lr=0.00806
Epoch 26/40 | lr=0.00800
Epoch 27/40 | lr=0.00794
Epoch 28/40 | lr=0.00787
Epoch 29/40 | lr=0.00781
Epoch 30/40 | lr=0.00775
Epoch 31/40 | lr=0.00769
Epoch 32/40 | lr=0.00763
Epoch 33/40 | lr=0.00758
Epoch 34/40 | lr=0.00752
Epoch 35/40 | lr=0.00746
Epoch 36/40 | lr=0.00741
Epoch 37/40 | lr=0.00735
Epoch 38/40 | lr=0.00730
Epoch 39/40 | lr=0.00725
Epoch 40/40 | lr=0.00719


In [11]:
def accuracy(y_pred, y_true):
    return np.mean(y_pred == y_true)


In [12]:
learning_rates = [0.01, 0.005]
epoch_values = [30, 40]
reg_values = [1e-4, 5e-4]

best_acc = 0
best_config = None
best_model = None

for lr in learning_rates:
    for ep in epoch_values:
        for reg in reg_values:

            print(f"\nTraining: lr={lr}, epochs={ep}, reg={reg}")

            model = Perceptron(alpha=lr, epochs=ep, decay=0.01, reg=reg)
            model.train(X_train, y_train)

            val_pred = model.predict(X_val)
            val_acc = accuracy(val_pred, y_val)

            print("Validation Accuracy:", val_acc)

            if val_acc > best_acc:
                best_acc = val_acc
                best_config = (lr, ep, reg)
                best_model = model

print("\nBest Config:", best_config)
print("Best Validation Accuracy:", best_acc)



Training: lr=0.01, epochs=30, reg=0.0001
Epoch 1/30 | lr=0.01000
Epoch 2/30 | lr=0.00990
Epoch 3/30 | lr=0.00980
Epoch 4/30 | lr=0.00971
Epoch 5/30 | lr=0.00962
Epoch 6/30 | lr=0.00952
Epoch 7/30 | lr=0.00943
Epoch 8/30 | lr=0.00935
Epoch 9/30 | lr=0.00926
Epoch 10/30 | lr=0.00917
Epoch 11/30 | lr=0.00909
Epoch 12/30 | lr=0.00901
Epoch 13/30 | lr=0.00893
Epoch 14/30 | lr=0.00885
Epoch 15/30 | lr=0.00877
Epoch 16/30 | lr=0.00870
Epoch 17/30 | lr=0.00862
Epoch 18/30 | lr=0.00855
Epoch 19/30 | lr=0.00847
Epoch 20/30 | lr=0.00840
Epoch 21/30 | lr=0.00833
Epoch 22/30 | lr=0.00826
Epoch 23/30 | lr=0.00820
Epoch 24/30 | lr=0.00813
Epoch 25/30 | lr=0.00806
Epoch 26/30 | lr=0.00800
Epoch 27/30 | lr=0.00794
Epoch 28/30 | lr=0.00787
Epoch 29/30 | lr=0.00781
Epoch 30/30 | lr=0.00775
Validation Accuracy: 0.304

Training: lr=0.01, epochs=30, reg=0.0005
Epoch 1/30 | lr=0.01000
Epoch 2/30 | lr=0.00990
Epoch 3/30 | lr=0.00980
Epoch 4/30 | lr=0.00971
Epoch 5/30 | lr=0.00962
Epoch 6/30 | lr=0.00952
Epoc

In [13]:
best_model = Perceptron(alpha=0.5, epochs=5, decay=0.001)
best_model.train(X_train, y_train)

y_test_pred = best_model.predict(X_test)
test_acc = np.mean(y_test_pred == y_test)

print("Test Accuracy:", test_acc)


Epoch 1/5 | lr=0.50000
Epoch 2/5 | lr=0.49950
Epoch 3/5 | lr=0.49900
Epoch 4/5 | lr=0.49850
Epoch 5/5 | lr=0.49801
Test Accuracy: 0.2706


In [14]:
pred_percept = best_model.predict(X_train)
print('The training accuracy is given by : %f' % (get_acc(pred_percept, y_train)))

The training accuracy is given by : 27.967347


### Validation

In [15]:
pred_percept = best_model.predict(X_val)
print('The validation accuracy is given by : %f' % (get_acc(pred_percept, y_val)))

The validation accuracy is given by : 28.200000


### Test Perceptron

In [16]:
pred_percept = best_model.predict(X_test)
print('The testing accuracy is given by : %f' % (get_acc(pred_percept, y_test)))

The testing accuracy is given by : 27.060000


In [17]:
y_test_pred = best_model.predict(X_test)


### Perceptron Kaggle Submission

Once you are satisfied with your solution and test accuracy output a file to submit your test set predictions to the Kaggle for Assignment 1 Perceptron. Use the following code to do so:

In [13]:
output_submission_csv('perceptron_submission.csv', y_test_pred)

# Support Vector Machines (with SGD)

Next, you will implement a "soft margin" SVM. In this formulation you will maximize the margin between positive and negative training examples and penalize margin violations using a hinge loss.

We will optimize the SVM loss using SGD. This means you must compute the loss function with respect to model weights. You will use this gradient to update the model weights.

SVM optimized with SGD has 3 hyperparameters that you can experiment with :
- **Learning rate** - similar to as defined above in Perceptron, this parameter scales by how much the weights are changed according to the calculated gradient update. 
- **Epochs** - similar to as defined above in Perceptron.
- **Regularization constant** - Hyperparameter to determine the strength of regularization. In this case it is a coefficient on the term which maximizes the margin.

You will implement the SVM using SGD in the **models/SVM.py**

The following code: 
- Creates an instance of the SVM classifier class 
- The train function of the SVM class is trained on the training data
- We use the predict function to find the training accuracy as well as the testing accuracy

### Train SVM

In [18]:
svm = SVM()
svm.train(X_train, y_train)

Epoch 1/50 | lr=0.01000
Epoch 2/50 | lr=0.00990
Epoch 3/50 | lr=0.00980
Epoch 4/50 | lr=0.00971
Epoch 5/50 | lr=0.00962
Epoch 6/50 | lr=0.00952
Epoch 7/50 | lr=0.00943
Epoch 8/50 | lr=0.00935
Epoch 9/50 | lr=0.00926
Epoch 10/50 | lr=0.00917
Epoch 11/50 | lr=0.00909
Epoch 12/50 | lr=0.00901
Epoch 13/50 | lr=0.00893
Epoch 14/50 | lr=0.00885
Epoch 15/50 | lr=0.00877
Epoch 16/50 | lr=0.00870
Epoch 17/50 | lr=0.00862
Epoch 18/50 | lr=0.00855
Epoch 19/50 | lr=0.00847
Epoch 20/50 | lr=0.00840
Epoch 21/50 | lr=0.00833
Epoch 22/50 | lr=0.00826
Epoch 23/50 | lr=0.00820
Epoch 24/50 | lr=0.00813
Epoch 25/50 | lr=0.00806
Epoch 26/50 | lr=0.00800
Epoch 27/50 | lr=0.00794
Epoch 28/50 | lr=0.00787
Epoch 29/50 | lr=0.00781
Epoch 30/50 | lr=0.00775
Epoch 31/50 | lr=0.00769
Epoch 32/50 | lr=0.00763
Epoch 33/50 | lr=0.00758
Epoch 34/50 | lr=0.00752
Epoch 35/50 | lr=0.00746
Epoch 36/50 | lr=0.00741
Epoch 37/50 | lr=0.00735
Epoch 38/50 | lr=0.00730
Epoch 39/50 | lr=0.00725
Epoch 40/50 | lr=0.00719
Epoch 41/

In [19]:
learning_rates = [0.01, 0.005, 0.001]
epochs_list = [40, 60, 80]
reg_constants = [0.05, 0.1, 0.2]


In [20]:
best_acc = 0
best_model = None
best_config = None

for lr in learning_rates:
    for ep in epochs_list:
        for reg in reg_constants:

            print(f"\nTraining SVM: lr={lr}, epochs={ep}, reg={reg}")

            svm = SVM(alpha=lr,
                      epochs=ep,
                      reg_const=reg,
                      batch_size=256,
                      decay=0.01)

            svm.train(X_train, y_train)

            val_pred = svm.predict(X_val)
            val_acc = np.mean(val_pred == y_val)

            print("Validation Accuracy:", val_acc)

            if val_acc > best_acc:
                best_acc = val_acc
                best_model = svm
                best_config = (lr, ep, reg)

print("\n===== BEST SVM CONFIG =====")
print("Best Hyperparameters:", best_config)
print("Best Validation Accuracy:", best_acc)



Training SVM: lr=0.01, epochs=40, reg=0.05
Epoch 1/40 | lr=0.01000
Epoch 2/40 | lr=0.00990
Epoch 3/40 | lr=0.00980
Epoch 4/40 | lr=0.00971
Epoch 5/40 | lr=0.00962
Epoch 6/40 | lr=0.00952
Epoch 7/40 | lr=0.00943
Epoch 8/40 | lr=0.00935
Epoch 9/40 | lr=0.00926
Epoch 10/40 | lr=0.00917
Epoch 11/40 | lr=0.00909
Epoch 12/40 | lr=0.00901
Epoch 13/40 | lr=0.00893
Epoch 14/40 | lr=0.00885
Epoch 15/40 | lr=0.00877
Epoch 16/40 | lr=0.00870
Epoch 17/40 | lr=0.00862
Epoch 18/40 | lr=0.00855
Epoch 19/40 | lr=0.00847
Epoch 20/40 | lr=0.00840
Epoch 21/40 | lr=0.00833
Epoch 22/40 | lr=0.00826
Epoch 23/40 | lr=0.00820
Epoch 24/40 | lr=0.00813
Epoch 25/40 | lr=0.00806
Epoch 26/40 | lr=0.00800
Epoch 27/40 | lr=0.00794
Epoch 28/40 | lr=0.00787
Epoch 29/40 | lr=0.00781
Epoch 30/40 | lr=0.00775
Epoch 31/40 | lr=0.00769
Epoch 32/40 | lr=0.00763
Epoch 33/40 | lr=0.00758
Epoch 34/40 | lr=0.00752
Epoch 35/40 | lr=0.00746
Epoch 36/40 | lr=0.00741
Epoch 37/40 | lr=0.00735
Epoch 38/40 | lr=0.00730
Epoch 39/40 | l

In [21]:
pred_svm = best_model.predict(X_train)
print('The training accuracy is given by : %f' % (get_acc(pred_svm, y_train)))

The training accuracy is given by : 43.538776


### Validate SVM

In [22]:
pred_svm = best_model.predict(X_val)
print('The validation accuracy is given by : %f' % (get_acc(pred_svm, y_val)))

The validation accuracy is given by : 40.700000


### Test SVM

In [23]:
pred_svm = best_model.predict(X_test)
print('The testing accuracy is given by : %f' % (get_acc(pred_svm, y_test)))

The testing accuracy is given by : 40.720000


### SVM Kaggle Submission

Once you are satisfied with your solution and test accuracy output a file to submit your test set predictions to the Kaggle for Assignment 1 SVM. Use the following code to do so:

In [25]:
output_submission_csv('svm_submission.csv', svm.predict(X_test))

# Softmax Classifier (with SGD)

Next, you will train a Softmax classifier. This classifier consists of a linear function of the input data followed by a softmax function which outputs a vector of dimension C (number of classes) for each data point. Each entry of the softmax output vector corresponds to a confidence in one of the C classes, and like a probability distribution, the entries of the output vector sum to 1. We use a cross-entropy loss on this sotmax output to train the model. 

Check the following link as an additional resource on softmax classification: http://cs231n.github.io/linear-classify/#softmax

Once again we will train the classifier with SGD. This means you need to compute the gradients of the softmax cross-entropy loss function according to the weights and update the weights using this gradient. Check the following link to help with implementing the gradient updates: https://deepnotes.io/softmax-crossentropy

The softmax classifier has 3 hyperparameters that you can experiment with :
- **Learning rate** - As above, this controls how much the model weights are updated with respect to their gradient.
- **Number of Epochs** - As described for perceptron.
- **Regularization constant** - Hyperparameter to determine the strength of regularization. In this case, we minimize the L2 norm of the model weights as regularization, so the regularization constant is a coefficient on the L2 norm in the combined cross-entropy and regularization objective.

You will implement a softmax classifier using SGD in the **models/Softmax.py**

The following code: 
- Creates an instance of the Softmax classifier class 
- The train function of the Softmax class is trained on the training data
- We use the predict function to find the training accuracy as well as the testing accuracy

### Train Softmax

In [24]:
softmax = Softmax()
softmax.train(X_train, y_train)

Epoch 1/100 | lr=0.50000
Epoch 2/100 | lr=0.49505
Epoch 3/100 | lr=0.49020
Epoch 4/100 | lr=0.48544
Epoch 5/100 | lr=0.48077
Epoch 6/100 | lr=0.47619
Epoch 7/100 | lr=0.47170
Epoch 8/100 | lr=0.46729
Epoch 9/100 | lr=0.46296
Epoch 10/100 | lr=0.45872
Epoch 11/100 | lr=0.45455
Epoch 12/100 | lr=0.45045
Epoch 13/100 | lr=0.44643
Epoch 14/100 | lr=0.44248
Epoch 15/100 | lr=0.43860
Epoch 16/100 | lr=0.43478
Epoch 17/100 | lr=0.43103
Epoch 18/100 | lr=0.42735
Epoch 19/100 | lr=0.42373
Epoch 20/100 | lr=0.42017
Epoch 21/100 | lr=0.41667
Epoch 22/100 | lr=0.41322
Epoch 23/100 | lr=0.40984
Epoch 24/100 | lr=0.40650
Epoch 25/100 | lr=0.40323
Epoch 26/100 | lr=0.40000
Epoch 27/100 | lr=0.39683
Epoch 28/100 | lr=0.39370
Epoch 29/100 | lr=0.39062
Epoch 30/100 | lr=0.38760
Epoch 31/100 | lr=0.38462
Epoch 32/100 | lr=0.38168
Epoch 33/100 | lr=0.37879
Epoch 34/100 | lr=0.37594
Epoch 35/100 | lr=0.37313
Epoch 36/100 | lr=0.37037
Epoch 37/100 | lr=0.36765
Epoch 38/100 | lr=0.36496
Epoch 39/100 | lr=0.3

In [25]:
learning_rates = [0.02, 0.01, 0.005, 0.001]
epochs_list = [50, 70, 90]
reg_constants = [0.05, 0.1, 0.2]

best_acc = 0
best_model = None
best_params = None

for lr in learning_rates:
    for ep in epochs_list:
        for reg in reg_constants:

            print(f"\nTraining Softmax: lr={lr}, epochs={ep}, reg={reg}")

            softmax = Softmax()
            softmax.alpha = lr
            softmax.epochs = ep
            softmax.reg_const = reg

            softmax.train(X_train, y_train)

            val_pred = softmax.predict(X_val)
            val_acc = get_acc(val_pred, y_val)

            print("Validation Accuracy:", val_acc)

            if val_acc > best_acc:
                best_acc = val_acc
                best_model = softmax
                best_params = (lr, ep, reg)

print("Best Learning Rate :", best_params[0])
print("Best Epochs        :", best_params[1])
print("Best Regularization:", best_params[2])
print("Best Validation Accuracy:", best_acc)



Training Softmax: lr=0.02, epochs=50, reg=0.05
Epoch 1/50 | lr=0.02000
Epoch 2/50 | lr=0.01980
Epoch 3/50 | lr=0.01961
Epoch 4/50 | lr=0.01942
Epoch 5/50 | lr=0.01923
Epoch 6/50 | lr=0.01905
Epoch 7/50 | lr=0.01887
Epoch 8/50 | lr=0.01869
Epoch 9/50 | lr=0.01852
Epoch 10/50 | lr=0.01835
Epoch 11/50 | lr=0.01818
Epoch 12/50 | lr=0.01802
Epoch 13/50 | lr=0.01786
Epoch 14/50 | lr=0.01770
Epoch 15/50 | lr=0.01754
Epoch 16/50 | lr=0.01739
Epoch 17/50 | lr=0.01724
Epoch 18/50 | lr=0.01709
Epoch 19/50 | lr=0.01695
Epoch 20/50 | lr=0.01681
Epoch 21/50 | lr=0.01667
Epoch 22/50 | lr=0.01653
Epoch 23/50 | lr=0.01639
Epoch 24/50 | lr=0.01626
Epoch 25/50 | lr=0.01613
Epoch 26/50 | lr=0.01600
Epoch 27/50 | lr=0.01587
Epoch 28/50 | lr=0.01575
Epoch 29/50 | lr=0.01562
Epoch 30/50 | lr=0.01550
Epoch 31/50 | lr=0.01538
Epoch 32/50 | lr=0.01527
Epoch 33/50 | lr=0.01515
Epoch 34/50 | lr=0.01504
Epoch 35/50 | lr=0.01493
Epoch 36/50 | lr=0.01481
Epoch 37/50 | lr=0.01471
Epoch 38/50 | lr=0.01460
Epoch 39/50

In [26]:
pred_softmax = best_model.predict(X_train)
print('The training accuracy is given by : %f' % (get_acc(pred_softmax, y_train)))

The training accuracy is given by : 35.002041


### Validate Softmax

In [27]:
pred_softmax = best_model.predict(X_val)
print('The validation accuracy is given by : %f' % (get_acc(pred_softmax, y_val)))

The validation accuracy is given by : 33.500000


### Testing Softmax

In [28]:
pred_softmax = best_model.predict(X_test)
print('The testing accuracy is given by : %f' % (get_acc(pred_softmax, y_test)))

The testing accuracy is given by : 32.320000


### Softmax Kaggle Submission

Once you are satisfied with your solution and test accuracy output a file to submit your test set predictions to the Kaggle for Assignment 1 Softmax. Use the following code to do so:

In [29]:
output_submission_csv('softmax_submission.csv', best_model.predict(X_test))